# 05 — Advanced Image Augmentation with Keras CV

Reference: [keras.io/keras_cv](https://keras.io/keras_cv)

**Runtime → T4 GPU**

| Technique | What it does |
|---|---|
| `RandAugment` | Randomly selects & applies N augmentation ops |
| `MixUp` | Blends two images + labels |
| `CutMix` | Pastes patch of one image onto another |
| `GridMask` | Masks grid regions of the image |
| `AugMix` | Applies random chains of augmentations |
| `RandomCutout` | Cuts out random rectangular regions |
| `Mosaic` | Combines 4 images into one grid |

In [ ]:
# Install keras-cv (pinned for Colab stability)
!pip install -q keras-cv tensorflow

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping
import keras_cv
import warnings; warnings.filterwarnings('ignore')

print('TF:', tf.__version__)
print('Keras CV:', keras_cv.__version__)

In [ ]:
# ── Dataset ──────────────────────────────────────────────────────────────────
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()
x_train = x_train.astype('float32')       # keras_cv expects 0-255 float or normalised
x_test  = x_test.astype('float32')
y_train = y_train.flatten().astype('int32')
y_test  = y_test.flatten().astype('int32')

# One-hot for MixUp / CutMix (they require one-hot labels)
y_train_oh = tf.one_hot(y_train, 10)
y_test_oh  = tf.one_hot(y_test,  10)

CLASS_NAMES = ['airplane','automobile','bird','cat','deer',
               'dog','frog','horse','ship','truck']
BATCH_SIZE  = 128
print(f'Train: {x_train.shape}  Test: {x_test.shape}')

In [ ]:
# Helper: show augmented grid
def show_augmented(aug_layer, x_sample, title, n=16, denorm=True):
    imgs = x_sample[:n]
    aug  = aug_layer(imgs, training=True)
    if isinstance(aug, dict):   # MixUp/CutMix return dict
        imgs_aug = aug['images']
    else:
        imgs_aug = aug
    imgs_aug = np.clip(imgs_aug.numpy() / 255.0, 0, 1)
    fig, axes = plt.subplots(2, n//2, figsize=(n, 4))
    for i, ax in enumerate(axes.flat):
        ax.imshow(imgs_aug[i]); ax.axis('off')
    plt.suptitle(title, fontsize=13)
    plt.tight_layout(); plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 1. RandAugment
# ══════════════════════════════════════════════════════════════════════════════
rand_aug = keras_cv.layers.RandAugment(
    value_range=(0, 255),
    augmentations_per_image=2,   # N ops per image
    magnitude=0.5,               # 0-1 strength
    magnitude_stddev=0.15
)
show_augmented(rand_aug, x_train, 'RandAugment (N=2, magnitude=0.5)')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 2. MixUp
# ══════════════════════════════════════════════════════════════════════════════
# MixUp works on batches with one-hot labels
mixup = keras_cv.layers.MixUp(alpha=0.4)

# Show MixUp on a batch
batch_imgs  = x_train[:16]
batch_labels = y_train_oh[:16]
mix_out = mixup({'images': batch_imgs, 'labels': batch_labels}, training=True)
mix_imgs = np.clip(mix_out['images'].numpy() / 255.0, 0, 1)

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(mix_imgs[i]); ax.axis('off')
plt.suptitle('MixUp (alpha=0.4) — note blended pixels', fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 3. CutMix
# ══════════════════════════════════════════════════════════════════════════════
cutmix = keras_cv.layers.CutMix(alpha=1.0)

batch_imgs   = x_train[:16]
batch_labels = y_train_oh[:16]
cm_out = cutmix({'images': batch_imgs, 'labels': batch_labels}, training=True)
cm_imgs = np.clip(cm_out['images'].numpy() / 255.0, 0, 1)

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(cm_imgs[i]); ax.axis('off')
plt.suptitle('CutMix — rectangular patch from another image', fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 4. GridMask
# ══════════════════════════════════════════════════════════════════════════════
grid_mask = keras_cv.layers.GridMask(
    ratio_factor=(0.3, 0.7),
    rotation_factor=(-0.1, 0.1)
)
show_augmented(grid_mask, x_train, 'GridMask — structured dropout of image regions')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 5. RandomCutout
# ══════════════════════════════════════════════════════════════════════════════
cutout = keras_cv.layers.RandomCutout(
    height_factor=(0.2, 0.4),
    width_factor=(0.2, 0.4)
)
show_augmented(cutout, x_train, 'RandomCutout (Cutout/Erasing)')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 6. AugMix
# ══════════════════════════════════════════════════════════════════════════════
augmix = keras_cv.layers.AugMix(value_range=(0, 255), severity=0.3)
show_augmented(augmix, x_train, 'AugMix — diverse stochastic augmentation chains')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 7. Standard geometric / colour layers in keras_cv
# ══════════════════════════════════════════════════════════════════════════════
geom_aug = keras.Sequential([
    keras_cv.layers.RandomFlip(mode='horizontal', bounding_box_format=None),
    keras_cv.layers.RandomRotation(factor=0.15, fill_mode='reflect'),
    keras_cv.layers.RandomZoom(height_factor=0.15, width_factor=0.15),
    keras_cv.layers.RandomShear(x_factor=0.1, y_factor=0.1),
    keras_cv.layers.RandomBrightness(factor=0.2, value_range=(0,255)),
    keras_cv.layers.RandomContrast(factor=0.2, value_range=(0,255)),
    keras_cv.layers.RandomSaturation(factor=(0.7, 1.3)),
], name='geometric_colour')
show_augmented(geom_aug, x_train, 'Geometric + Colour Augmentation Pipeline')

In [ ]:
# ── Build model + full keras-cv pipeline ────────────────────────────────────
def build_model():
    inp = keras.Input(shape=(32, 32, 3))
    x = layers.Rescaling(1.0/255)(inp)   # normalise inside model
    for f in [32, 64, 128]:
        x = layers.Conv2D(f, 3, padding='same', activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.MaxPooling2D()(x)
        x = layers.Dropout(0.25)(x)
    x   = layers.GlobalAveragePooling2D()(x)
    x   = layers.Dense(256, activation='relu')(x)
    x   = layers.Dropout(0.4)(x)
    out = layers.Dense(10, activation='softmax')(x)
    return keras.Model(inp, out)

def make_tf_dataset(x, y_oh, augment_fn=None, batch_size=128, shuffle=True):
    ds = tf.data.Dataset.from_tensor_slices({'images': x, 'labels': y_oh})
    if shuffle:
        ds = ds.shuffle(10000)
    ds = ds.batch(batch_size)
    if augment_fn:
        ds = ds.map(lambda batch: augment_fn(batch, training=True),
                    num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.map(lambda batch: (batch['images'], batch['labels']),
                num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

print('Model and dataset factory ready.')

In [ ]:
# ── A/B Test: No aug vs RandAugment vs CutMix+MixUp ────────────────────────
# CutMix → MixUp pipeline (applied to batches with one-hot labels)
cutmix_mixup_fn = keras.Sequential([cutmix, mixup])

def make_dataset_cutmix_mixup(x, y_oh, batch_size=128, shuffle=True):
    ds = tf.data.Dataset.from_tensor_slices({'images': x, 'labels': y_oh})
    if shuffle:
        ds = ds.shuffle(10000)
    ds = ds.batch(batch_size)
    ds = ds.map(lambda b: cutmix({'images': b['images'], 'labels': b['labels']}, training=True),
                num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.map(lambda b: mixup({'images': b['images'], 'labels': b['labels']}, training=True),
                num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.map(lambda b: (b['images'], b['labels']), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

# Simple augmentation for baseline aug
def simple_aug_fn(batch, training=True):
    if training:
        batch['images'] = rand_aug(batch['images'], training=True)
    return batch

train_no_aug   = make_tf_dataset(x_train, y_train_oh, augment_fn=None)
train_randaug  = make_tf_dataset(x_train, y_train_oh, augment_fn=simple_aug_fn)
train_cutmix   = make_dataset_cutmix_mixup(x_train, y_train_oh)
test_ds        = make_tf_dataset(x_test,  y_test_oh, augment_fn=None, shuffle=False)

print('Datasets created.')

In [ ]:
EPOCHS = 25
cv_results = {}

for ds_name, train_ds in [('No Augmentation', train_no_aug),
                           ('RandAugment',     train_randaug),
                           ('CutMix+MixUp',    train_cutmix)]:
    print(f'\nTraining: {ds_name} ...')
    m = build_model()
    # For soft labels (MixUp/CutMix), use categorical crossentropy
    m.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])
    h = m.fit(train_ds, epochs=EPOCHS,
              validation_data=test_ds,
              callbacks=[EarlyStopping(patience=5, restore_best_weights=True)],
              verbose=0)
    cv_results[ds_name] = h
    _, acc = m.evaluate(test_ds, verbose=0)
    print(f'  Test accuracy: {acc:.4f}')

In [ ]:
# Plot keras-cv A/B comparison
colors = ['#e74c3c','#3498db','#9b59b6']
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for (name, h), c in zip(cv_results.items(), colors):
    axes[0].plot(h.history['val_accuracy'], label=name, color=c)
    axes[1].plot(h.history['val_loss'],     label=name, color=c)

for ax, title in zip(axes, ['Val Accuracy', 'Val Loss']):
    ax.set_title(f'Keras CV A/B Test — {title}')
    ax.set_xlabel('Epoch'); ax.legend(); ax.grid(True, alpha=0.3)
plt.suptitle('No Aug vs RandAugment vs CutMix+MixUp', fontsize=12, y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# ── Side-by-side visualisation: all techniques on same image ─────────────────
sample_batch = {'images': x_train[:8], 'labels': y_train_oh[:8]}
techs = {
    'Original'   : lambda b: b['images'],
    'RandAugment': lambda b: rand_aug(b['images'], training=True),
    'GridMask'   : lambda b: grid_mask(b['images'], training=True),
    'Cutout'     : lambda b: cutout(b['images'], training=True),
    'AugMix'     : lambda b: augmix(b['images'], training=True),
    'CutMix'     : lambda b: cutmix(b, training=True)['images'],
    'MixUp'      : lambda b: mixup(b, training=True)['images'],
}

fig, axes = plt.subplots(len(techs), 8, figsize=(16, len(techs)*2))
for row, (tname, tfn) in enumerate(techs.items()):
    imgs = np.clip(tfn(sample_batch).numpy() / 255.0, 0, 1)
    for col in range(8):
        axes[row, col].imshow(imgs[col]); axes[row, col].axis('off')
    axes[row, 0].set_ylabel(tname, fontsize=10, rotation=0, labelpad=60, va='center')
plt.suptitle('Keras CV Augmentation Techniques Side-by-Side', fontsize=14, y=1.01)
plt.tight_layout(); plt.show()